# Debug: #619 split 3/3 — gap-edge formula change

Verifies the deferred patch in `compute_plausible_gaps`:

```python
# before (current master):
h_textlines[i].x0 - h_textlines[i - 1].x0   # left-edge-to-left-edge stride
# after:
h_textlines[i].x0 - h_textlines[i - 1].x1   # actual whitespace gap
```

**Setup:** `pip install -e .[plot]` in the branch worktree, then run this notebook.

**Expected outcome of this branch:** the xfailed tests in `tests/test_network.py` (test_issue_585, test_issue_585_network_flavor_with_table_areas) flip from `xfail` → `xpass`. No other test fixtures change.

**Reproducers used:**
- `tests/files/multiple_tables.pdf` (#585 original report)
- `tests/files/good_energy.pdf` (added in #745)

Issue: [#770](https://github.com/camelot-dev/camelot/issues/770) · Tracking PR (TODO marker only): #771

## ⚙️ Colab bootstrap (run me first)

On Google Colab this clones the `debug/619-split3-gap-formula` branch, installs camelot editable, and cd-s in. On a local checkout it is a no-op.


In [ ]:
import sys, os, subprocess

BRANCH = "debug/619-split3-gap-formula"
REPO = "https://github.com/bosd/camelot.git"

if "google.colab" in sys.modules:
    if os.path.basename(os.getcwd()) != "camelot" and not os.path.isdir("camelot/.git"):
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO], check=True)
    if os.path.basename(os.getcwd()) != "camelot":
        os.chdir("camelot")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[plot]", "pytest"], check=True)
    print("Colab bootstrap complete -> cwd:", os.getcwd())
else:
    print("Not on Colab - assuming a local editable checkout; skipping bootstrap.")


## What this notebook does (plain version)

**The bug (#585):** the `network` parser measures the gap between text
blocks as *left-edge to left-edge* (the stride between two words) instead
of the *actual whitespace* (right edge of one word → left edge of the
next). On some PDFs this makes it miss columns/tables.

**The fix:** a 2-character change in `camelot/parsers/network.py` —
`compute_plausible_gaps` should use `.x1` (right edge) of the previous
textline instead of `.x0`.

**What you'll do here:** 4 cells, in order —
1. bootstrap (already above): clone + install.
2. show the two lines we're about to change.
3. apply the 2-char patch to the file on disk.
4. run the tests: the two `#585` tests should pass, and the rest of the
   network + hybrid suite should stay green.

If step 4 is all green, the fix is good — open a PR and you're done.
Tracking issue: #770.

### 1. Show the two lines we're going to change

In [ ]:
import subprocess
print(subprocess.run(
    ['grep', '-n', '-A1',
     'textlines\\[i\\].x0 - \\|textlines\\[i\\].y0 - ',
     'camelot/parsers/network.py'],
    capture_output=True, text=True).stdout)
# You should see two spots:
#   h_textlines[i].x0 - h_textlines[i - 1].x0
#   v_textlines[i].y0 - v_textlines[i - 1].y0

### 2. Apply the patch
Changes `.x0` → `.x1` (and `.y0` → `.y1`) on the *previous* textline only.
Re-runnable: if already patched, the replace is a no-op.

In [ ]:
p = 'camelot/parsers/network.py'
s = open(p).read()
before = s
s = s.replace('h_textlines[i].x0 - h_textlines[i - 1].x0',
              'h_textlines[i].x0 - h_textlines[i - 1].x1')
s = s.replace('v_textlines[i].y0 - v_textlines[i - 1].y0',
              'v_textlines[i].y0 - v_textlines[i - 1].y1')
open(p, 'w').write(s)
print('patched!' if s != before else 'no change (already patched)')
# Confirm:
import subprocess
print(subprocess.run(['grep','-n','x1\\|y1','camelot/parsers/network.py'],
                     capture_output=True, text=True).stdout[:400])

### 3. Run the two #585 tests
These are currently marked `xfail` (expected-to-fail). With the patch they
should now **pass** (you'll see `XPASS` or, if you remove the markers,
`PASSED`). `-rX` shows xpasses explicitly.

In [ ]:
import subprocess
r = subprocess.run(
    ['python','-m','pytest',
     'tests/test_network.py::test_issue_585',
     'tests/test_network.py::test_issue_585_network_flavor_with_table_areas',
     '-v','-rX','--no-header'],
    capture_output=True, text=True)
print(r.stdout[-2500:])

### 4. Regression sweep — the whole network + hybrid suite
**This is the verdict.** If everything here passes, the fix is safe to ship.
If something *other than* the two #585 tests fails, the gap-formula change
disturbed another fixture — paste me the failure and we'll decide whether
to update the fixture or refine the fix.

In [ ]:
import subprocess
r = subprocess.run(
    ['python','-m','pytest','tests/test_network.py','tests/test_hybrid.py',
     '-q','--no-header'],
    capture_output=True, text=True)
print(r.stdout[-4000:])

### If step 4 is green
You're done validating. Two finishing touches for the real PR (I can do
these for you — just say so):

1. Remove the `@pytest.mark.xfail(...)` decorators above `test_issue_585`
   and `test_issue_585_network_flavor_with_table_areas` in
   `tests/test_network.py` (they pass now, so they shouldn't be marked
   expected-fail).
2. Remove the `TODO(#619)` comment block in `compute_plausible_gaps`.

Then commit the patched `network.py` + `test_network.py` and open a PR
that says *Closes #770, #585*.